In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path

#
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Core.logging import flow_logger as logger
from Instruments.GX_271.dim import DIM
from Instruments.WPI_Syr_pump.Pump import AL1000
from Instruments.Runze_valves.Runze62Valve import Runze62Valve
from Ecosystems.reaction_segment_generation import RSG

In [2]:
# --- Instantiate serial object for AL1000 pump ----
ser = serial.Serial("COM2", 9600, timeout=0.5)
pump = AL1000(ser, device_address="@00", sleep_time=0.5)

pump.connect()

Device is connected


In [3]:
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Start logging (declare this run belongs to the experiment "Development") ---
logger.start_experiment("Development")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()
#DIM = DIM()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=36,
    y_offset=10,
    x_min=27,
    x_max=127,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=201,
    y_offset=39,
    x_min=157,
    x_max=247,
    y_min=2,
    y_max=292
) 

# Instantiate the RSG ecosystem with the connected Gilson and AL1000
rsg = RSG(gilson=g, pump=pump, syringe_diameter=4.606)

In [4]:
g.home()

All axes homed successfully and Z is within safe limits


In [155]:
g.go_into_vial("rack2", 2)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(201.0), np.float64(106.0), 20)

In [6]:
g.move_z(50)

'Moved Z to 50. Result: No valid response received.'

In [156]:

rate = 0.5           # mL/min
volume = 200           # uL
direction = "INF"     # WDR = withdraw, INF = Infuse

pump.prepare(rate=rate, volume=volume, direction=direction)

Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL200
Sent: @00*DIRINF


In [158]:
pump.start()

Sent: @00*RUN 1


'00I'

In [145]:
rsg.take_air_gap(10)

Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [146]:
rsg.prepickup()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [147]:
rsg.take_air_gap(5)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [148]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(36.0), np.float64(10.0))

In [149]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [150]:
rsg.pickup_from_vial("rack1", 1, 50)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL50
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [151]:
g.go_to_vial("rack1", 19)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(52.55), np.float64(54.5675))

In [154]:
g.move_z(130)

'Moved Z to 130. Result: Move Z: 2, Success'

In [153]:
rsg.dispense_in_vial("rack1", 19, 75)

#Note - 50 + 5 + 10 + 10 (final 10 is 10uL of the 40uL air gap, leaving 30uL)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL75
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP


In [4]:
rsg.pickup_from_vial("rack2", 1, 200)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL200
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [6]:
rsg.prepickup()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL10
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [7]:
rsg.take_air_gap(5)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [8]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130


(np.float64(36.0), np.float64(10.0))

In [9]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [10]:
rsg.pickup_from_vial("rack1", 1, 50)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL50
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP


In [111]:
g.move_z(130)

'Moved Z to 130. Result: Move Z: 2, Success'

In [114]:
g.go_to_vial("rack1", 19)
g.move_z(119)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


'Moved Z to 119. Result: Move Z: 2, Success'

In [13]:
rsg.dispense_in_vial("rack1", 19, 70)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL70
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP


In [14]:
g.move_z(130)

'Moved Z to 130. Result: Move Z: 2, Success'

In [15]:
rsg.wash_sequence()

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.05 MM
Sent: @00*VOL5.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL100.0
Sent: @00*DIRWDR
Sent: @00*RUN 1
Sent: @00*STP
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
Sent: @00*PHN1
Sent: @00*RAT C 0.5 MM
Sent: @00*VOL105.0
Sent: @00*DIRINF
Sent: @00*RUN 1
Sent: @00*STP


In [14]:
g.move_z(130)

'Moved Z to 130. Result: Move Z: 2, Success'

In [29]:
pump.start()

Sent: @00*RUN 1


'00I'

In [15]:
g.go_to_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130
[DEBUG] ensure_z_safe: current_z=130.0 is already >= target_z=130.0


(np.float64(201.0), np.float64(39.0))

In [8]:
rsg.dispense_in_vial("rack1", 18, 70)

[DEBUG] ensure_z_safe: current_z=119.0 is already >= target_z=119.0

--- Preparing pump (Phase 1) ---
Sent: @00*PHN1
Sent: @00*DIA4.606
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*RAT C 0.5 MM
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*VOL70
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*DIRINF
Raw reply: 00S
Parsed reply: Acknowledged = True
--- Preparation complete ---

Sent: @00*RUN 1
Sent: @00*STP


In [16]:
rsg.wash_sequence()

[autoreload of Ecosystems.reaction_segment_generation failed: Traceback (most recent call last):
  File "C:\Users\CHAD-HPLC\anaconda3\envs\flowctrl\Lib\site-packages\IPython\extensions\autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "C:\Users\CHAD-HPLC\anaconda3\envs\flowctrl\Lib\site-packages\IPython\extensions\autoreload.py", line 580, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "C:\Users\CHAD-HPLC\anaconda3\envs\flowctrl\Lib\importlib\__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 936, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1074, in get_code
  File "<frozen importlib._bootstrap_external>", line 1004, in source_to_code
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "C:\Users\CHAD-HPLC\Documents\VictorFlow\Ecosys

TypeError: RSG.wash_sequence() missing 4 required positional arguments: 'solvent_module', 'solvent_vial', 'target_module', and 'target_vial'

In [10]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=120.0 is already >= target_z=119.0
[DEBUG] ensure_z_safe: current_z=120.0 is already >= target_z=119.0


(np.float64(36.25), np.float64(10.0))

In [23]:
syringe_dia = 4.606    # mm
rate = 0.050            # mL/min
volume = 200           # uL
direction = "WDR"     # WDR = withdraw, INF = Infuse

pump.prepare(dia=syringe_dia, rate=rate, volume=volume, direction=direction)


--- Preparing pump (Phase 1) ---
Sent: @00*PHN1
Sent: @00*DIA4.606
DIA
Raw reply: 00P
Sent: @00*RAT C 0.05 MM
RAT
Raw reply: 00P
Sent: @00*VOL200
VOL
Raw reply: 00P
Sent: @00*DIRWDR
Raw reply: 00S
Parsed reply: Acknowledged = True
--- Preparation complete ---



In [11]:
g.go_to_vial("rack2",1)

[DEBUG] ensure_z_safe: current_z=125.0 is already >= target_z=119
[DEBUG] ensure_z_safe: current_z=125.0 is already >= target_z=119.0


(np.float64(201.0), np.float64(39.0))

In [24]:
pump.start()

Sent: @00*RUN 1


'00W'

In [21]:
pump.stop()

Sent: @00*STP


'00A?S'

In [16]:
pump.set_volume(108.5)
pump.set_direction("INF")
pump.set_rate(0.5)

Sent: @00*VOL108.5
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*DIRINF
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*RAT C 0.5 MM
Raw reply: 00S
Parsed reply: Acknowledged = True


'00S'

In [22]:
g.go_into_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=119.0 is already >= target_z=119


(np.float64(201.0), np.float64(39.0), 20)

In [18]:
g.move_z(30)

'Moved Z to 30. Result: Move Z: 2, Success'

In [4]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [3]:
g.go_to_vial("rack1", 2)

[DEBUG] ensure_z_safe: current_z=119.0 is already >= target_z=119.0
[DEBUG] ensure_z_safe: current_z=119.0 is already >= target_z=119.0


(np.float64(36.25), np.float64(27.827))

In [9]:
g.move_z(45)

'Moved Z to 45. Result: Move Z: 2, Success'

In [11]:
pump.set_volume(65)
pump.set_direction("INF")
pump.set_rate(0.5)

Sent: @00*VOL65
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*DIRINF
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*RAT C 0.5 MM
Raw reply: 00S
Parsed reply: Acknowledged = True


'00S'

In [12]:
g.go_into_vial("rack1", 17)

[DEBUG] ensure_z_safe: current_z=45.0 is already >= target_z=45.0
[DEBUG] ensure_z_safe: current_z=45.0 is already >= target_z=45.0


(np.float64(52.55), np.float64(17.3135), 11.0)

In [13]:
g.go_into_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=96.0 is already >= target_z=93
[DEBUG] ensure_z_safe: current_z=96.0 is already >= target_z=93


(np.float64(201.0), np.float64(39.0), 1)

In [13]:
pump.start()

Sent: @00*RUN 1


'00I'

In [16]:
pump.set_volume(5)

Sent: @00*VOL5
Raw reply: 00S
Parsed reply: Acknowledged = True


'00S'

In [17]:
pump.start()

Sent: @00*RUN 1


'00W'

In [13]:
g.go_into_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=119.0 is already >= target_z=119


(np.float64(36.0), np.float64(8.4), 87.0)

In [12]:
g.move_y(10)

[DEBUG] ensure_z_safe: current_z=119.0 is already >= target_z=119.0


'Moved Y to 10. Result: Move Y: 2, Success'

In [5]:
print(g.current_y)

8.4


In [13]:
g.send_command("Get XY Position")

'Get XY Position: 36.25, 10'

In [14]:
g.home()

Z below safe limit (87.00 < 125.00) — raising first.
All axes homed successfully and Z is within safe limits


In [8]:
g.go_to_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=119.0 is already >= target_z=93
[DEBUG] ensure_z_safe: current_z=119.0 is already >= target_z=119.0


(np.float64(201.0), np.float64(39.0))

In [8]:
pump.start()

Sent: @00*RUN 1


'00W'

In [2]:
# --- Instantiate the runze valve --- 

runze = Runze62Valve(
    port="COM7",
    baudrate=9600,
    timeout=1
)

runze.connect()

#valve.go_to_pos(1)
#valve.go_to_pos(2)
#valve.toggle()


Connected to Runze valve on COM7
Valve moved to position 1


In [3]:
g.home()

All axes homed successfully and Z is within safe limits


In [3]:
runze.read_pos()

1

In [9]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=45.0
[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=93


(np.float64(36.5), np.float64(7.6))

In [5]:
time.sleep(5)
g.go_into_vial("rack2", 1)
time.sleep(2)
pump.start()

[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=93
[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=93
Sent: @00*RUN 1


'00W'

In [6]:
g.go_to_vial("rack1", 1)
g.move_z(45)

syringe_dia = 4.606    # mm
rate = 0.5            # mL/min
volume = 200           # uL
direction = "INF"     # WDR = withdraw, INF = Infuse

pump.prepare(dia=syringe_dia, rate=rate, volume=volume, direction=direction)

pump.start()

[DEBUG] ensure_z_safe: current_z=93.0 is already >= target_z=93

--- Preparing pump (Phase 1) ---
Sent: @00*PHN1
Sent: @00*DIA4.606
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*RAT C 0.5 MM
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*VOL200
Raw reply: 00S
Parsed reply: Acknowledged = True
Sent: @00*DIRINF
Raw reply: 00S
Parsed reply: Acknowledged = True
--- Preparation complete ---

Sent: @00*RUN 1


'00I'

In [16]:
g.move_z(20, allow_in_vial=True)

'Moved Z to 20. Result: Move Z: 2, Success'

In [14]:
g.send_command("Get XY Position")

'Get XY Position: 35.6, 8.1'

In [5]:
g.move_z(119)

'Moved Z to 119. Result: Move Z: 2, Success'

In [34]:
g.move_y(7.6)

[DEBUG] ensure_z_safe: current_z=45.0 is already >= target_z=45.0


'Moved Y to 7.6. Result: Move Y: 2, Success'

In [14]:
g.go_to_vial("rack1", 1)

[DEBUG] ensure_z_safe: current_z=125.0 is already >= target_z=45.0
[DEBUG] ensure_z_safe: current_z=125.0 is already >= target_z=93


(np.float64(36.0), np.float64(8.4))

In [9]:
print(g.current_z)

119.0


In [22]:
g.move_x(36)

[DEBUG] ensure_z_safe: current_z=45.0 is already >= target_z=45.0


'Moved X to 36. Result: Move X: 2, Success'

In [19]:
g.move_y(10)

[DEBUG] ensure_z_safe: current_z=119.0 is already >= target_z=45.0


'Moved Y to 10. Result: Move Y: 2, Success'

In [17]:
print(g.current_y)

8.4


In [10]:
g.go_to_vial("rack2", 1)

[DEBUG] ensure_z_safe: current_z=93 is already >= target_z=45.0


(np.float64(201.0), np.float64(39.0))

In [11]:
print(g.current_module)

rack1
